In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import numpy as np
import json

import utils.functions as f

import datagenerator.generators.synthesizer as synth

import warnings
warnings.filterwarnings("ignore", message="Warning")

2025-10-20 13:35:21.519008: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/nlv19419/.pyenv/versions/incartdb-venv/lib/python3.11/site-packages/torch/_functorch/deprecated.py:61: UserWarning: We've integrated functorch into PyTorch. As the final step of the integration, functorch.vmap is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use torch.vmap instead; see the PyTorch 2.0 release notes and/or the torch.func migration guide for more details https://pytorch.org/docs/master/func.migrating.html
  warn_deprecated('vmap', 'torch.vmap')


# Overview

The plan for the HC2 analysis in the Experimenter is to show that its correction of existing bias can be accomplished using synthetic data as well (at least 🤞🏻 to a certain degree). To this end, fit a multivariate sampler to the data that were prepared by Joseph and myself and then execute polarized sampling for a particular, to be determined, feature.

The idea is that, first, the original data set is analyzed to show a particular disparate impact. This DI will then be repaired and the pre/post comparison of the Experimenter will show that the Fairness metric increased as a result of the repair. Next, synthetic data is created to complement the original data to in this way reduce the disbalance on the relevant feature. Then repair is not needed and the Fairness metrics should be ok from the get go.

In this way, this notebook can also sere as example of how to use the multivariate sampler.

# Data overview

The original demographics from the incart database has been extended with an additional feature that combines gender and diagnosis to more clearly show that there a different prevalences for particular diagnoses comparing men and women.


In [4]:
dataset = "incartdb_demographics_expanded_gxd"
df = pd.read_csv(f"{dataset}.csv", sep=",")
df.head(10)

,age,gender,pid,diagnosis,present,gxd,age_bin
0,65,F,patient 1,Coronary artery disease,YES,F_Coronary artery disease,60-70
1,65,F,patient 1,arterial hypertension,YES,F_arterial hypertension,60-70
2,65,F,patient 1,Acute MI,NO,F_Acute MI,60-70
3,65,F,patient 1,Transient ischemic attack,NO,F_Transient ischemic attack,60-70
4,65,F,patient 1,left ventricular hypertrophy,NO,F_left ventricular hypertrophy,60-70
5,65,F,patient 1,Earlier MI,NO,F_Earlier MI,60-70
6,65,F,patient 1,Sinus node dysfunction,NO,F_Sinus node dysfunction,60-70
7,59,M,patient 2,Coronary artery disease,NO,M_Coronary artery disease,50-60
8,59,M,patient 2,arterial hypertension,NO,M_arterial hypertension,50-60
9,59,M,patient 2,Acute MI,YES,M_Acute MI,50-60


# Polarized sampling

To create (additional, complementary) data that has improved balance, first, a multivariate model needs to be fitted from which, in a second step, conditional samples can be drawn. For the present case that only serves as illustration of the principle, we only address a single diagnosis; Note that if multiple conditions are required, as many polarized samples need to be "manually" drawn and concatenated into a data frame.

In [8]:
df[df["present"]=="YES"].drop_duplicates(subset=['pid', 'diagnosis'])['gxd'].value_counts(normalize=False)

gxd
F_Coronary artery disease         6
F_arterial hypertension           6
M_Transient ischemic attack       4
F_left ventricular hypertrophy    3
M_Earlier MI                      3
M_Acute MI                        2
F_Sinus node dysfunction          1
F_Earlier MI                      1
M_Coronary artery disease         1
M_arterial hypertension           1
M_left ventricular hypertrophy    1
Name: count, dtype: int64

In [9]:
df['diagnosis'].value_counts()

diagnosis
Coronary artery disease         32
arterial hypertension           32
Acute MI                        32
Transient ischemic attack       32
left ventricular hypertrophy    32
Earlier MI                      32
Sinus node dysfunction          32
Name: count, dtype: int64

In [10]:
sampler = synth.MultivariateSampler(data=df.drop(columns=["gxd"]), 
                                    numerical_features=["age"], 
                                    categorical_features=["pid", "gender", "diagnosis", "present", "age_bin"])
sampler.fit()

In [11]:
new_sample = pd.DataFrame()

for gender in ["F", "M"]:
    for diagn in ["Coronary artery disease", "arterial hypertension", "Acute MI",
                  "Transient ischemic attack", "left ventricular hypertrophy", "Earlier MI",
                  "Sinus node dysfunction"]:
        s = sampler.sample(10, conditions={"gender": gender, "diagnosis": diagn, "present": "YES"})
        new_sample = pd.concat([s, new_sample])

new_sample["pid"] = "p" + new_sample.index.astype(str)
new_sample["gxd"] = new_sample["gender"].astype(str) + "_" + new_sample["diagnosis"].astype(str)
    
new_sample.head(10)

,age,gender,pid,diagnosis,present,age_bin,gxd
0,74.322893,M,p0,Sinus node dysfunction,YES,70-80,M_Sinus node dysfunction
1,43.532305,M,p1,Sinus node dysfunction,YES,40-50,M_Sinus node dysfunction
2,82.049979,M,p2,Sinus node dysfunction,YES,70-80,M_Sinus node dysfunction
3,42.842736,M,p3,Sinus node dysfunction,YES,30-40,M_Sinus node dysfunction
4,64.162513,M,p4,Sinus node dysfunction,YES,60-70,M_Sinus node dysfunction
5,54.346105,M,p5,Sinus node dysfunction,YES,50-60,M_Sinus node dysfunction
6,62.680972,M,p6,Sinus node dysfunction,YES,60-70,M_Sinus node dysfunction
7,69.030109,M,p7,Sinus node dysfunction,YES,70-80,M_Sinus node dysfunction
8,56.173055,M,p8,Sinus node dysfunction,YES,50-60,M_Sinus node dysfunction
9,64.978393,M,p9,Sinus node dysfunction,YES,60-70,M_Sinus node dysfunction


In [14]:
df_synth = pd.concat([df[df["present"]=="NO"], new_sample], ignore_index=True)
df_synth.head(15)

,age,gender,pid,diagnosis,present,gxd,age_bin
0,65.0,F,patient 1,Acute MI,NO,F_Acute MI,60-70
1,65.0,F,patient 1,Transient ischemic attack,NO,F_Transient ischemic attack,60-70
2,65.0,F,patient 1,left ventricular hypertrophy,NO,F_left ventricular hypertrophy,60-70
3,65.0,F,patient 1,Earlier MI,NO,F_Earlier MI,60-70
4,65.0,F,patient 1,Sinus node dysfunction,NO,F_Sinus node dysfunction,60-70
5,59.0,M,patient 2,Coronary artery disease,NO,M_Coronary artery disease,50-60
6,59.0,M,patient 2,arterial hypertension,NO,M_arterial hypertension,50-60
7,59.0,M,patient 2,Transient ischemic attack,NO,M_Transient ischemic attack,50-60
8,59.0,M,patient 2,left ventricular hypertrophy,NO,M_left ventricular hypertrophy,50-60
9,59.0,M,patient 2,Earlier MI,NO,M_Earlier MI,50-60


In [15]:
df_synth.to_csv(dataset + "_complemented.csv", index=False, encoding='UTF-8')